# CF Anchor Correlation Diagnosis

Does the denoiser's counterfactual-slot prediction actually track aux_outcome's per-subject cf_target, or is it largely unresponsive to x?

In [1]:
import json
from pathlib import Path

import torch

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded
from src.model import HybridModel

/home/justin/msc_ai/individual-project/diffusion-irregular-ehr/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TAU_FIXED = 100  # fixed mid-schedule tau

In [3]:
# Trained HybridModel endpoint with cf_anchor_weight > 0
run_id = "hybrid_conf_2026-09-03T11_15_30_rep1"

results_json_path = Path(f"results/results_{run_id}.json")
with open(results_json_path) as f:
    results = json.load(f)
cfg = Config.model_validate(results["config"])
print("Counterfactual anchor weight:", cfg.diffusion.cf_anchor_weight)
ckpt_path = Path(cfg.train.checkpoint_dir) / f"final_model_{run_id}.pth"

torch.manual_seed(cfg.train.seed)
_, val_ds, _, _ = load_ihdp(
    cfg.data.path,
    replication=cfg.data.replication,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)
val_ds = make_ihdp_confounded(val_ds, effect=cfg.data.confounder_effect)

model = HybridModel(cfg.model, cfg.diffusion)
model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=False))
model.eval()

# Take a=1 subjects -- their Y(0) slot is counterfactual
mask = val_ds.a.bool()
x, a, y_fac = val_ds.x[mask], val_ds.a[mask], val_ds.y[mask]
n = x.shape[0]
print(f"N subjects (a=1): {n}")

Counterfactual anchor weight: 1.0
N subjects (a=1): 75


In [9]:
torch.manual_seed(0)
with torch.no_grad():
    cf_target = model.aux_outcome.mean(x, 1.0 - a)  # what the anchor supervises toward

    z, _, _ = model.encoder.rsample(x, a, y_fac)  # matches compute_loss's conditioning

    y_both = model._assemble_yboth(a, y_fac, cf_target)  # (N,2)
    tau = torch.full((n,), TAU_FIXED, dtype=torch.long)
    eps = torch.randn(n, 2)
    ab_tau = model.alpha_bar_sched[tau].unsqueeze(1)
    noisy_y = ab_tau.sqrt() * y_both + (1.0 - ab_tau).sqrt() * eps

    # Run through real trained denoiser and invert to get clean prediction
    eps_pred = model.denoiser(noisy_y, tau, z, a)
    clean_pred = (noisy_y - (1.0 - ab_tau).sqrt() * eps_pred) / ab_tau.sqrt()

# Y(0) is index 0 in _assemble_yboth's [y(0),y(1)] layout; for a=1 subjects this is
# the counterfactual slot
clean_pred_y0 = clean_pred[:, 0]

# Compare clean estimate with aux_outcome's cf_target
corr = torch.corrcoef(torch.stack([clean_pred_y0, cf_target]))[0, 1]

print(f"cf_target std across subjects: {cf_target.std().item()}")
print(f"cf_target range: {cf_target.min().item()} {cf_target.max().item()}")

print(f"\nclean_pred (Y0, counterfactual) std across subjects: {clean_pred_y0.std().item()}")
print(f"clean_pred_y0 range: {clean_pred_y0.min().item()} {clean_pred_y0.max().item()}")

print(f"\ncorrelation(clean_pred_y0, cf_target): {corr.item()}")

cf_target std across subjects: 0.5217007398605347
cf_target range: -1.286789894104004 1.061202049255371

clean_pred (Y0, counterfactual) std across subjects: 1.2711505889892578
clean_pred_y0 range: -3.113527297973633 2.373953342437744

correlation(clean_pred_y0, cf_target): 0.3805086612701416


A model that's actually tracking the target should show high correlation and comparable spread; unresponsive-to-x collapse shows near-zero spread; noisy/unstable fitting shows weak correlation with excess spread.